# 03 — Оценка downstream-задач

Четыре класса задач из раздела 6 статьи TwHIN, на замороженных эмбеддингах:
1. **Candidate generation** (friend recommendation) — HitRate@K, Recall@K, MRR;
2. **Engagement ranking** (review edges) — ROC-AUC, PR-AUC, RCE;
3. **Content probe** (useful) — PR-AUC, ROC-AUC.

В конце — таблица сравнения с числами из статьи.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())
assert (Path.cwd() / "src").is_dir()

# какой чекпоинт оцениваем (по умолчанию последний best.pt; можно указать конкретный tag)
CKPT = "outputs/checkpoints/all__ns.pt"
assert Path(CKPT).exists(), f"нет чекпоинта {CKPT} — сначала запусти ноутбук 02"

cwd: /home/nikita/DL/recsys/twhin_yelp_2


## 1. Candidate generation (friend recommendation)

Аналог Who-to-Follow. Метрики должны быть **заметно выше** sanity-бейзлайнов
(random / popular) — иначе эмбеддинги ничего не выучили.

In [2]:
out = !python -m src.evaluation.retrieval --config configs/base.yaml --ckpt {CKPT}
print("\n".join(out))

Candidate generation (friend recommendation)
  queries evaluated: 46912
     K |  HitRate@K |   Recall@K |   random |  popular
    10 |     0.0062 |     0.0020 |   0.0001 |   0.0158
    20 |     0.0108 |     0.0036 |   0.0003 |   0.0272
    50 |     0.0219 |     0.0077 |   0.0005 |   0.0511
  MRR: 0.0028


## 2. Engagement ranking (review edges)

Бинарное предсказание engagement, негативы — случайные бизнесы (не все сущности!).

In [3]:
out = !python -m src.evaluation.engagement --config configs/base.yaml --ckpt {CKPT}
print("\n".join(out))

Engagement ranking (review edges, business negatives)
  test size:   118992
  prevalence:  0.5000
  ROC-AUC:     0.6031
  PR-AUC:      0.5795
  RCE:         1.81


## 3. Content probe (useful)

PR-AUC сравнивается с `prevalence` (доля позитивов) — это baseline случайного предсказателя.

In [4]:
out = !python -m src.evaluation.content_probe --config configs/base.yaml --ckpt {CKPT}
print("\n".join(out))

Useful/content probe (label: useful>0)
  test size:   59496
  prevalence:  0.3592  (PR-AUC baseline)
  PR-AUC:      0.4716
  ROC-AUC:     0.6304


## 4. Сравнение со статьёй TwHIN

Числа статьи приведены как **ориентир порядка величины** (у нас Yelp, не Twitter,
и линейные пробинг-модели вместо production-ранкеров — абсолюты совпадать не обязаны;
важна воспроизводимость **направления** эффекта). Заполни `ours_*` своими результатами.

In [ ]:
import pandas as pd

# Точные ориентиры из статьи TwHIN (KDD 2022). Открытые Yelp-числа будут
# отличаться по абсолюту; нас интересует воспроизведение НАПРАВЛЕНИЯ эффектов.
comparison = pd.DataFrame([
    {"task": "Candidate gen R@10 (Table 1)",
     "paper_unimodal": "0.58%", "paper_mixture": "3.70%",
     "note": "mixture >> unimodal (>300% lift) — главный эффект Sec 4.4"},
    {"task": "Candidate gen R@20 (Table 1)",
     "paper_unimodal": "1.02%", "paper_mixture": "5.53%", "note": ""},
    {"task": "Candidate gen R@50 (Table 1)",
     "paper_unimodal": "2.06%", "paper_mixture": "8.79%", "note": ""},
    {"task": "Ads ranking RCE (Table 2)",
     "paper_unimodal": "baseline 13-21", "paper_mixture": "+U/+A/+T растёт",
     "note": "online +2.38 RCE, -10.3% cost/conv; U даёт основной прирост"},
    {"task": "Search MAP / ROC (Table 3)",
     "paper_unimodal": "55.7 / 57.9", "paper_mixture": "57.0 / 59.6",
     "note": "Uf,Ue помогают; author A — только в комбинации с user"},
    {"task": "Offensive PR-AUC (Table 4)",
     "paper_unimodal": "0.41-0.47 (text)", "paper_mixture": "+TwHIN 0.52",
     "note": "+9.09% rel на Collection1; нейтрально на Collection2"},
])
comparison

\* Числа статьи — приблизительные ориентиры; сверь точные значения по PDF статьи
(метрики и абсолюты у авторов на приватных данных Twitter). Для отчёта важнее
**направление**: гетерогенность (ablation `all` vs `review_only`) улучшает метрики,
а sampled-softmax+logQ не хуже ns. Это проверяется прогоном 03 на разных чекпоинтах.